In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

from typing import Any, Tuple, List, Dict
import warnings
import datetime
from pathlib import Path
import os

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import statsmodels.api as sm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api

Missing data:
- Directly queryable from db
  - Centroid and ID tracking over the full period of time
  - Foraging bouts
- To be generated from raw data
  - Sleeping
  - Exploring bouts

# Load data

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "pre_social_start": '2024-01-31 11:00:00', "pre_social_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "post_social_start": '2024-02-25 16:00:00', "post_social_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "pre_social_start": '2024-01-31 10:00:00', "pre_social_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 12:00:00', "post_social_start": '2024-02-25 16:00:00', "post_social_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "pre_social_start": '2024-06-08 18:00:00', "pre_social_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 10:00:00', "social_end": '2024-07-06 13:00:00', "post_social_start": '2024-07-07 15:00:00', "post_social_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "pre_social_start": '2024-06-08 18:00:00', "pre_social_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 11:00:00', "social_end": '2024-07-03 14:00:00', "post_social_start": '2024-07-04 10:00:00', "post_social_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "pre_social_start": '2024-08-16 16:00:00', "pre_social_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 13:00:00', "post_social_start": '2024-09-09 17:00:00', "post_social_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "pre_social_start": '2024-08-16 14:00:00', "pre_social_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 09:00:00', "social_end": '2024-09-09 01:00:00', "post_social_start": '2024-09-09 14:00:00', "post_social_end": '2024-09-22 16:00:00'}
]

## Patch data

In [ ]:

def load_subject_patch_data(
    key: dict[str, str],
    period_start: str,
    period_end: str
) -> tuple[list[dict[str, str]], pd.DataFrame]:
    """Loads subject patch data for a specified time period.

    Args:
        key (dict): The key to filter the subject patch data.
        period_start (str): The start time for the period.
        period_end (str): The end time for the period.

    Returns:
        tuple: A tuple containing:
            - patch_info (list of dict): Information about patches.
            - block_subject_patch_data (pd.DataFrame): Data for the specified period.
    """
    patch_info = (
        BlockAnalysis.Patch
        & key
        & f'block_start >= "{period_start}"'
        & f'block_start <= "{period_end}"'
    ).fetch('block_start', "patch_name", "patch_rate", "patch_offset", as_dict=True)

    block_subject_patch_data = (
        BlockSubjectAnalysis.Patch() 
        & key 
        & f'block_start >= "{period_start}"' 
        & f'block_start <= "{period_end}"'
    ).fetch(format="frame")

    if not block_subject_patch_data.empty:
        block_subject_patch_data.reset_index(level=["experiment_name"], drop=True, inplace=True) 
        block_subject_patch_data.reset_index(inplace=True)

    return patch_info, block_subject_patch_data

In [ ]:
patch_info_dict = {}
subject_patch_data_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Define periods
    periods = {
        "pre_social": (exp["pre_social_start"], exp["pre_social_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "post_social": (exp["post_social_start"], exp["post_social_end"])
    }

    # Initialize nested dictionaries for this experiment
    patch_info_dict[exp["name"]] = {}
    subject_patch_data_dict[exp["name"]] = {}

    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        # Convert string dates to datetime if needed
        period_start = datetime.strptime(period_start, "%Y-%m-%d %H:%M:%S")
        period_end = datetime.strptime(period_end, "%Y-%m-%d %H:%M:%S")

        # Load data for this period
        patch_info, block_subject_patch_data = load_subject_patch_data(
            key, period_start, period_end
        )

        # Filter out dummy patches
        if not block_subject_patch_data.empty:
            block_subject_patch_data = block_subject_patch_data[
                ~block_subject_patch_data["patch_name"].str.contains("PatchDummy")
            ]

            # Add experiment name as a column
            block_subject_patch_data.insert(0, "experiment_name", exp["name"])

            # For pre-social and post-social periods check n_subjects per block (should == 1)
            if period_name in ["pre_social", "post_social"]:
                n_subjects = (
                    block_subject_patch_data.groupby("block_start")["subject_name"].nunique()
                )
                if (n_subjects != 1).any():
                    warnings.warn(
                        f"Pre or post social data for {exp['name']} has blocks with more than one "
                        f"subject being tracked. Data needs to be fixed or cleaned."
                    )

        # Store the data
        patch_info_dict[exp["name"]][period_name] = patch_info
        subject_patch_data_dict[exp["name"]][period_name] = block_subject_patch_data

# Combine data across experiments for each period
combined_data = {}
for period_name in ["pre_social", "social", "post_social"]:
    period_data = []
    for exp_name in patch_info_dict:
        if not subject_patch_data_dict[exp_name][period_name].empty:
            period_data.append(subject_patch_data_dict[exp_name][period_name])

    if period_data:
        combined_data[period_name] = pd.concat(period_data)
    else:
        # Create empty DataFrame with expected columns if no data
        combined_data[period_name] = pd.DataFrame()

# Assign to the original variable names for compatibility
block_subject_patch_data_pre_social_combined = combined_data["pre_social"]
block_subject_patch_data_social_combined = combined_data["social"]
block_subject_patch_data_post_social_combined = combined_data["post_social"]

# Display one of the dataframes as an example
block_subject_patch_data_social_combined

### Foraging bouts

In [ ]:
def load_foraging_bouts(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """
    Loads foraging bout data for blocks falling within a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period (format: '%Y-%m-%d %H:%M:%S').
        period_end (str): End datetime of the time period (format: '%Y-%m-%d %H:%M:%S').

    Returns:
        pd.DataFrame: Concatenated dataframe of foraging bouts for all matching blocks.
                      Returns an empty dataframe with predefined columns if no data found.
    """
    # Fetch block start times within the specified period
    blocks = (
        Block
        & key
        & f"block_start >= '{period_start}'"
        & f"block_end <= '{period_end}'"
    ).fetch("block_start")

    # Retrieve foraging bouts for each block
    bouts = []
    for block_start in blocks:
        block_key = key | {"block_start": str(block_start)}
        bouts.append(get_foraging_bouts(block_key))

    # Return concatenated DataFrame or empty fallback
    if bouts:
        return pd.concat(bouts, ignore_index=True)
    else:
        return pd.DataFrame(
            columns=["start", "end", "n_pellets", "cum_wheel_dist", "subject"]
        )

In [ ]:
foraging_pre_social_dict = {}
foraging_social_dict = {}
foraging_post_social_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Load foraging bout data for each time period
    pre_df = load_foraging_bouts(key, exp["pre_social_start"], exp["pre_social_end"])
    social_df = load_foraging_bouts(key, exp["social_start"], exp["social_end"])
    post_df = load_foraging_bouts(key, exp["post_social_start"], exp["post_social_end"])

    # Add experiment name as a column
    pre_df.insert(0, "experiment_name", exp["name"])
    social_df.insert(0, "experiment_name", exp["name"])
    post_df.insert(0, "experiment_name", exp["name"])

    # Store individual DataFrames in dictionaries
    foraging_pre_social_dict[exp["name"]] = pre_df
    foraging_social_dict[exp["name"]] = social_df
    foraging_post_social_dict[exp["name"]] = post_df

# Combine all foraging bout data
foraging_pre_social = pd.concat(foraging_pre_social_dict.values(), ignore_index=True)
foraging_social = pd.concat(foraging_social_dict.values(), ignore_index=True)
foraging_post_social = pd.concat(foraging_post_social_dict.values(), ignore_index=True)

# Final formatting
for df in (foraging_pre_social, foraging_social, foraging_post_social):
    df.sort_values(["experiment_name", "start"], inplace=True)
    df.reset_index(drop=True, inplace=True)

# Display an example
foraging_social

## SLEAP tracking data

In [ ]:
def load_position_data(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads position data (centroid tracking) for a specified time period.
    
    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period.
        period_end (str): End datetime of the time period.
        
    Returns:
        pd.DataFrame: DataFrame containing position data for the specified period.
                     Returns an empty DataFrame if no data found.
    """
    try:
        print(f"  Querying data from {period_start} to {period_end}...")
        
        # Create chunk restriction for the time period
        chunk_restriction = acquisition.create_chunk_restriction(key["experiment_name"], period_start, period_end)
        
        # Create the query
        pose_query = (
            streams.SpinnakerVideoSource
            * tracking.SLEAPTracking.PoseIdentity.proj(
                "identity_name", "identity_likelihood", "anchor_part"
            )
            * tracking.SLEAPTracking.AnchorPart
            & key
            & {
                "spinnaker_video_source_name": "CameraTop",
            }
            & chunk_restriction
        )
        
        # Fetch the data
        centroid_df = fetch_stream(pose_query)
        
        # Clean up the dataframe
        if not centroid_df.empty:
            if "spinnaker_video_source_name" in centroid_df.columns:
                centroid_df.drop(columns=["spinnaker_video_source_name"], inplace=True)
            
            # Add experiment name column for reference
            centroid_df.insert(0, "experiment_name", key["experiment_name"])
            
            print(f"  Retrieved {len(centroid_df)} rows of position data")
        else:
            print("  No data found for the specified period")
        
        return centroid_df
    
    except Exception as e:
        print(f"  Error loading position data for {key['experiment_name']} ({period_start} to {period_end}): {e}")
        return pd.DataFrame()  # Empty DataFrame

def save_position_data_to_parquet(
    df: pd.DataFrame, 
    experiment_name: str, 
    period_name: str,
    data_dir: Path = Path("/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")
) -> Path:
    """Saves position data DataFrame to a parquet file.
    
    Args:
        df (pd.DataFrame): Position data to save
        experiment_name (str): Name of the experiment
        period_name (str): Period name (pre_social, social, post_social)
        data_dir (Path): Directory to save the file (default: "/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")
        
    Returns:
        Path: Path to the saved file
    """
    # Create directory if it doesn't exist
    os.makedirs(data_dir, exist_ok=True)
    
    # Add period column for reference
    df = df.copy()
    df = df.reset_index()
    df["period"] = period_name
    
    # Create filename
    filename = f"{experiment_name}_{period_name}_position.parquet"
    file_path = data_dir / filename
    
    print(f"  Saving to {file_path}...")
    # Save to parquet with compression
    df.to_parquet(file_path, compression="snappy")
    
    # Report file stats
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"  Saved successfully: {len(df)} rows, {memory_usage_mb:.2f} MB in memory, {file_size_mb:.2f} MB on disk")
    
    return file_path

def load_position_data_from_parquet(
    experiment_name: str = None,
    period: str = None,
    data_dir: Path = Path("/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")
) -> pd.DataFrame:
    """Loads saved position data from parquet files.
    
    Args:
        experiment_name (str, optional): Filter by experiment name. If None, load all experiments.
        period (str, optional): Filter by period (pre_social, social, post_social). If None, load all periods.
        data_dir (Path): Directory containing parquet files (default: "/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")
        
    Returns:
        pd.DataFrame: Combined DataFrame of all matching parquet files.
    """
    if not data_dir.exists():
        print(f"Directory {data_dir} does not exist. No position data files found.")
        return pd.DataFrame()
    
    # Create pattern based on filters
    pattern = ""
    if experiment_name:
        pattern += f"{experiment_name}_"
    else:
        pattern += "*_"
        
    if period:
        pattern += f"{period}_"
    else:
        pattern += "*_"
        
    pattern += "position.parquet"
    
    # Find matching files
    matching_files = list(data_dir.glob(pattern))
    
    if not matching_files:
        print(f"No matching position data files found with pattern: {pattern}")
        return pd.DataFrame()
    
    print(f"Found {len(matching_files)} matching files")
    
    # Load and concatenate matching files
    dfs = []
    total_rows = 0
    for file in matching_files:
        print(f"Loading {file}...")
        df = pd.read_parquet(file)
        total_rows += len(df)
        dfs.append(df)
        print(f"  Loaded {len(df)} rows")
    
    # Combine data
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        combined_df = combined_df.set_index('time')
        print(f"Combined data: {len(combined_df)} rows")
        return combined_df
    else:
        return pd.DataFrame()

In [ ]:
# Directory to save parquet files
data_dir = Path("/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")
os.makedirs(data_dir, exist_ok=True)

# Process all experiments and all time periods
for exp in experiments:
    key = {"experiment_name": exp["name"]}
    print(f"\nProcessing experiment: {exp['name']}")
    
    # Define time periods
    periods = {
        "pre_social": (exp["pre_social_start"], exp["pre_social_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "post_social": (exp["post_social_start"], exp["post_social_end"])
    }
    
    # Process each period
    for period_name, (period_start, period_end) in periods.items():
        print(f"\n  Loading {period_name} position data...")
        
        # Load position data for this period
        position_df = load_position_data(key, period_start, period_end)
        
        if not position_df.empty:
            # Save to parquet
            save_position_data_to_parquet(
                position_df, 
                exp["name"], 
                period_name,
                data_dir
            )
        else:
            print(f"  No position data to save for {exp['name']} during {period_name} period")

In [ ]:
# Example 1: Load all pre-social data
pre_social_df = load_position_data_from_parquet(period="pre_social")
display(pre_social_df.head())

# Example 2: Load just one experiment's social data
social_exp3_df = load_position_data_from_parquet(
    experiment_name="social0.2-aeon3", 
    period="pre_social"
)
display(social_exp3_df.head())

### Sleep bouts

### Exploring bouts